In [1]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 1. .env 파일의 GOOGLE_API_KEY 로드
load_dotenv(override=True)

# 2. Gemini 임베딩 모델 선언
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
query_result = embeddings.embed_query("저는 배가 고파요")
print("벡터 차원 수:", len(query_result))  #3072차원
print("임베딩 결과 (일부):", query_result[:5])

# 3. 샘플 코퍼스 데이터프레임 생성
data = [
    "저는 배가 고파요",
    "저는 배가 엄청 불러요",
    "저기 배가 지나가네요",
    "굶어서 허기가 지네요",
    "허기 위기라는 게임이 있는데 즐거워",
    "스팀에서 재밌는 거 해야지",
    "스팀에어프라이어에 연어구이 해먹을거야"
]

df = pd.DataFrame(data, columns=["text"])

# 4. 전체 코퍼스 배치 임베딩 (1회 API 호출)
df["embedding"] = embeddings.embed_documents(df["text"].tolist())
print(df)

# 5. 코사인 유사도 함수 정의
def cos_sim(a, b):
    return dot(a, b) / (norm(a) * norm(b))

# 6. 의미 기반 유사 문서 검색 함수 정의
def return_answer_candidate(df: pd.DataFrame, query: str, top_k: int = 3):
    # (1) 질의어 벡터화
    query_vec = embeddings.embed_query(query)

    # (2) 각 문서 벡터와의 코사인 유사도 연산
    df["similarity"] = df["embedding"].apply(
        lambda doc_vec: cos_sim(np.array(doc_vec), np.array(query_vec))
    )

    # (3) 유사도 내림차순 정렬 후 상위 k개 문서 반환
    return df.sort_values(by="similarity", ascending=False).head(top_k)

# 7. 의미 검색 테스트
query_text = "아무것도 안 먹었더니 꼬르륵 소리가 나네"
sim_result = return_answer_candidate(df, query_text, top_k=3)

print(f"검색 질의: \"{query_text}\"\n")
print("--- [검색 결과 Top 3] ---")
print(sim_result[["text", "similarity"]].to_string(index=False))

벡터 차원 수: 3072
임베딩 결과 (일부): [0.0010935758, 0.021031871, 0.02997788, -0.07138116, -0.011080982]
                   text                                          embedding
0             저는 배가 고파요  [-0.013088046, 0.02243476, 0.027823744, -0.076...
1          저는 배가 엄청 불러요  [-0.0030146784, 0.017104417, 0.02869768, -0.08...
2           저기 배가 지나가네요  [-0.02218078, 0.020155584, 0.01889986, -0.0963...
3           굶어서 허기가 지네요  [-0.008105187, 0.021042852, 0.010521777, -0.07...
4   허기 위기라는 게임이 있는데 즐거워  [0.005046692, 0.008384599, 0.008965597, -0.065...
5        스팀에서 재밌는 거 해야지  [-0.014958823, 0.011937631, 0.002658975, -0.08...
6  스팀에어프라이어에 연어구이 해먹을거야  [0.005578815, 0.010928705, 0.029554935, -0.084...
검색 질의: "아무것도 안 먹었더니 꼬르륵 소리가 나네"

--- [검색 결과 Top 3] ---
        text  similarity
 굶어서 허기가 지네요    0.769414
저는 배가 엄청 불러요    0.741716
   저는 배가 고파요    0.741290
